# Gradient Descent (GD) 

This notebook applies the systematic Lyapunov-function discovery procedure to gradient descent with step size $1/L$:
$x_{k+1}=x_k-\frac{1}{L}\nabla f(x_k)$, for convex and $L$-smooth minimization. Its tight rate was first proved in 
"Performance of first-order methods for smooth convex minimization: A novel approach" by Drori and Teboulle (2014), 
and was later presented in a more elementary form in "An elementary approach to tight worst case complexity analysis
of gradient based methods" by Teboulle and Vaisbourd (2023).


## Import the required libraries

In [1]:
import pepflow as pf
import numpy as np
import sympy as sp
import itertools
from IPython.display import display, Math

import pepflow.lyapunov_utils as lu

## Define the function class

In [2]:
L = pf.Parameter("L")
f = pf.SmoothConvexFunction(is_basis=True, tags=["f"], L=L)

## Write a function that generates a PEPContext object associated with GD

In [3]:
def make_ctx_gd(
    ctx_name: str, N: int | sp.Integer, stepsize: pf.Parameter
) -> pf.PEPContext:
    ctx_gd = pf.PEPContext(ctx_name).set_as_current()
    x = pf.Vector(is_basis=True, tags=["x_0"])
    f.set_stationary_point("x_star")

    for i in range(int(N)):
        x = x - stepsize * f.grad(x)
        x.add_tag(f"x_{i + 1}")

    return ctx_gd

## Numerical PEP solve 
### First, find tight and sparsified proof certificates:

In [4]:
N = 5
N_int = int(N)
R = pf.Parameter("R")
L_value = 1
R_value = 1

ctx_prf = make_ctx_gd(ctx_name="ctx_prf", N=N, stepsize=1 / L)
pb_prf = pf.PEPBuilder(ctx_prf)
pb_prf.add_initial_constraint(
    ((ctx_prf["x_0"] - ctx_prf["x_star"]) ** 2).le(R, name="initial_condition")
)
pb_prf.set_performance_metric(f(ctx_prf[f"x_{N}"]) - f(ctx_prf["x_star"]))

result_dense = pb_prf.solve(resolve_parameters={"L": L_value, "R": R_value})
lamb_dense = result_dense.get_scalar_constraint_dual_value_in_numpy(f)

print("dense optimum:", result_dense.opt_value)

dense optimum: 0.04545413917119227


In [5]:
def tag_to_index(tag: str, N: int | sp.Basic = N) -> int:
    if tag == "x_star":
        return int(N) + 1
    return int(tag.split("_", 1)[1])


def gd_relaxed_constraints_from_observed_pattern(lamb_matrix, N: int | sp.Integer):
    relaxed = []
    N = int(N)
    for tag_i in lamb_matrix.row_names:
        i = tag_to_index(tag_i, N)
        if i == N + 1:
            continue
        for tag_j in lamb_matrix.col_names:
            j = tag_to_index(tag_j, N)
            if i < N and i + 1 == j:
                continue
            relaxed.append(f"f:{tag_i},{tag_j}")
    return relaxed


pb_prf.set_relaxed_constraints(
    gd_relaxed_constraints_from_observed_pattern(lamb_dense, N)
)
result = pb_prf.solve(resolve_parameters={"L": L_value, "R": R_value})

# Dual variable associated with the initial condition.
tau_sol = result.dual_var_manager.dual_value("initial_condition")
# Dual variables associated with the interpolation conditions of f.
lamb_sol = result.get_scalar_constraint_dual_value_in_numpy(f)
# Dual matrix associated with the Gram matrix.
S_sol = result.get_gram_dual_matrix()

print("sparse optimum:", result.opt_value)
print("tau:", tau_sol)

sparse optimum: 0.04545448086517327
tau: 0.04545454544805585


In [6]:
lamb_sol.pprint()

<IPython.core.display.Math object>

In [7]:
S_sol.pprint()

<IPython.core.display.Math object>

### Perform the reverse-basis LDL decomposition on semidefinite matrix certificate $S$
### The order of the PEP basis vectors is reversed to $[\nabla f(x_N),\ldots,\nabla f(x_0),x_\star,x_0]$.

In [8]:
pm = pf.ExpressionManager(ctx_prf, resolve_parameters={"L": L_value, "R": R_value})
x = ctx_prf.tracked_point(f)
x_0 = ctx_prf["x_0"]
x_star = ctx_prf["x_star"]

D, ell = lu.ldl_decompose_with_reversed_basis(
    S_sol, ctx_prf.basis_vectors(), print_output=True
)

diag_D = np.diag(D)
print(
    "rank(S):", np.linalg.matrix_rank(np.asarray(S_sol.matrix, dtype=float), tol=1e-7)
)
print("diag(D):", diag_D)
print("nonzero diagonal entries:", np.where(np.abs(diag_D) > 1e-7)[0])

<IPython.core.display.Math object>

Matrix([
[0.5,   0.0,   0.0,   0.0,   0.0,   0.0, 0.0, 0.0],
[0.0, 0.819,   0.0,   0.0,   0.0,   0.0, 0.0, 0.0],
[0.0,   0.0, 0.541,   0.0,   0.0,   0.0, 0.0, 0.0],
[0.0,   0.0,   0.0, 0.336,   0.0,   0.0, 0.0, 0.0],
[0.0,   0.0,   0.0,   0.0, 0.179,   0.0, 0.0, 0.0],
[0.0,   0.0,   0.0,   0.0,   0.0, 0.055, 0.0, 0.0],
[0.0,   0.0,   0.0,   0.0,   0.0,   0.0, 0.0, 0.0],
[0.0,   0.0,   0.0,   0.0,   0.0,   0.0, 0.0, 0.0]])

rank(S): 6
diag(D): [ 5.00000002e-01  8.19445133e-01  5.40816781e-01  3.35937847e-01
  1.79012431e-01  5.50000144e-02  1.77538106e-16 -1.03349710e-16]
nonzero diagonal entries: [0 1 2 3 4 5]


### Numerical values of $D = \mathrm{diag}(\alpha_0, \dots, \alpha_5, *, *)$

In [9]:
# Store the numerical square coefficients and vectors in chronological order:
# sos_vectors[i] corresponds to the square term involving data through x_i.
sos_coeffs = [D[N_int - i, N_int - i] for i in range(N_int + 1)]
sos_vectors = [ell[N_int - i] for i in range(N_int + 1)]

for i, coeff in enumerate(sos_coeffs):
    print(f"alpha_{i} = {coeff:.12g}")

alpha_0 = 0.0550000143791
alpha_1 = 0.179012431223
alpha_2 = 0.335937847388
alpha_3 = 0.540816781461
alpha_4 = 0.819445132815
alpha_5 = 0.500000002236


---

## Step 1. Propose a candidate Lyapunov function

### For $k=0,\dots,N$, take $\mathcal I_k=\{(i-1,i):i=1,\ldots,k\}\cup\{(\star,i):i=0,\ldots,k\},\,\, \mathcal J_k=\{0,\ldots,k\}$ and let $V_{-1}=0$

In [10]:
def named_matrix_entry(named_matrix, row_name: str, col_name: str):
    row = named_matrix.row_names.index(row_name)
    col = named_matrix.col_names.index(col_name)
    return named_matrix.matrix[row, col]


def lamb_numeric(tag_i: str, tag_j: str):
    return named_matrix_entry(lamb_sol, tag_i, tag_j)


V_raw = [pf.Scalar.zero()]
partial_sum = pf.Scalar.zero()

partial_sum += lamb_numeric("x_star", "x_0") * f.interp_ineq("x_star", "x_0")
partial_sum -= sos_coeffs[0] * sos_vectors[0] ** 2
V_raw.append(partial_sum)

for step in range(N_int):
    partial_sum += lamb_numeric(f"x_{step}", f"x_{step + 1}") * f.interp_ineq(
        f"x_{step}", f"x_{step + 1}"
    )
    partial_sum += lamb_numeric("x_star", f"x_{step + 1}") * f.interp_ineq(
        "x_star", f"x_{step + 1}"
    )
    partial_sum -= sos_coeffs[step + 1] * sos_vectors[step + 1] ** 2
    V_raw.append(partial_sum)

print("V_raw contains:", [f"V_{i - 1}" for i in range(len(V_raw))])

V_raw contains: ['V_-1', 'V_0', 'V_1', 'V_2', 'V_3', 'V_4', 'V_5']


## Step 2. Check for admissibility

### Sufficiency is immediate: $\mathcal I_N=\mathcal I$ and $\mathcal J_N=\mathcal J$. 
### We check for consistency and conciseness:

In [11]:
scalar_labels = [str(s) for s in ctx_prf.basis_scalars()]

for idx, V in enumerate(V_raw):
    k_label = idx - 1
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(V_eval.inner_prod_coords.astype(float), tol=1e-6)
    func_coords = V_eval.func_coords.astype(float)
    support = [scalar_labels[i] for i, val in enumerate(func_coords) if abs(val) > 1e-7]
    print(f"V_{k_label}: rank={rank}, nonzero function coordinates={support}")

print("\nFunction-value vector at last iteration:")
pf.pprint_labeled_vector(
    pm.eval_scalar(V_raw[N + 1]).func_coords.astype(float),
    scalar_labels,
)

V_-1: rank=0, nonzero function coordinates=[]
V_0: rank=2, nonzero function coordinates=['f(x_star)', 'f(x_0)']
V_1: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_1)']
V_2: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_2)']
V_3: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_3)']
V_4: rank=3, nonzero function coordinates=['f(x_star)', 'f(x_4)']
V_5: rank=1, nonzero function coordinates=['f(x_star)', 'f(x_5)']

Function-value vector at last iteration:


<IPython.core.display.Math object>

## Step 3. Build a set of special candidate vectors

In [12]:
ctx_prf.set_as_current()
lyap_basis_candidate = list(ctx_prf.basis_vectors())
lyap_basis_candidate += x[1 : N_int + 1]

base_count = len(lyap_basis_candidate)
for i, j in itertools.combinations(range(base_count), 2):
    diff = lyap_basis_candidate[i] - lyap_basis_candidate[j]
    lyap_basis_candidate.append(diff)

print(lyap_basis_candidate)

[x_0, x_star, grad_f(x_0), grad_f(x_1), grad_f(x_2), grad_f(x_3), grad_f(x_4), grad_f(x_5), x_1, x_2, x_3, x_4, x_5, x_0-x_star, x_0-grad_f(x_0), x_0-grad_f(x_1), x_0-grad_f(x_2), x_0-grad_f(x_3), x_0-grad_f(x_4), x_0-grad_f(x_5), x_0-(x_1), x_0-(x_2), x_0-(x_3), x_0-(x_4), x_0-(x_5), x_star-grad_f(x_0), x_star-grad_f(x_1), x_star-grad_f(x_2), x_star-grad_f(x_3), x_star-grad_f(x_4), x_star-grad_f(x_5), x_star-(x_1), x_star-(x_2), x_star-(x_3), x_star-(x_4), x_star-(x_5), grad_f(x_0)-grad_f(x_1), grad_f(x_0)-grad_f(x_2), grad_f(x_0)-grad_f(x_3), grad_f(x_0)-grad_f(x_4), grad_f(x_0)-grad_f(x_5), grad_f(x_0)-(x_1), grad_f(x_0)-(x_2), grad_f(x_0)-(x_3), grad_f(x_0)-(x_4), grad_f(x_0)-(x_5), grad_f(x_1)-grad_f(x_2), grad_f(x_1)-grad_f(x_3), grad_f(x_1)-grad_f(x_4), grad_f(x_1)-grad_f(x_5), grad_f(x_1)-(x_1), grad_f(x_1)-(x_2), grad_f(x_1)-(x_3), grad_f(x_1)-(x_4), grad_f(x_1)-(x_5), grad_f(x_2)-grad_f(x_3), grad_f(x_2)-grad_f(x_4), grad_f(x_2)-grad_f(x_5), grad_f(x_2)-(x_1), grad_f(x_2)-(x_

## Step 4. Find a basis of $\mathbf V_k$ within the candidate vectors

### We observe that all $V_k$ ($k=1,\dots,N-1$) contains a term $-\tau\|x_0-x_\star\|^2$, where $\tau=\tau_N$ is the value of the worst-case performance metric.

In [13]:
for idx, V in enumerate(V_raw):
    print(
        f"V_{idx - 1}:",
        lu.vectors_in_column_space(
            V,
            lyap_basis_candidate,
            ctx_prf,
            resolve_parameters=pm.resolve_parameters,
            rtol=1e-5,
            atol=1e-5,
        ),
    )

V_-1: []
V_0: [grad_f(x_0), x_0-x_star, x_0-(x_1), x_star-(x_1)]
V_1: [grad_f(x_0), grad_f(x_1), x_0-x_star, x_0-(x_1), x_0-(x_2), x_star-(x_1), x_star-(x_2), grad_f(x_0)-grad_f(x_1), x_1-(x_2)]
V_2: [grad_f(x_2), x_0-x_star, x_0-(x_2), x_0-(x_3), x_star-(x_2), x_star-(x_3), x_2-(x_3)]
V_3: [grad_f(x_3), x_0-x_star, x_0-(x_3), x_0-(x_4), x_star-(x_3), x_star-(x_4), x_3-(x_4)]
V_4: [grad_f(x_4), x_0-x_star, x_0-(x_4), x_0-(x_5), x_star-(x_4), x_star-(x_5), x_4-(x_5)]
V_5: [x_0-x_star]


In [14]:
for idx in range(2, len(V_raw) - 1):
    aligned_special_vectors = lu.vectors_in_column_space(
        V_raw[idx],
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        atol=1e-5,
    )
    best_vectors, best_coefficients = lu.find_basis_with_sparsest_coefficients(
        V_raw[idx],
        aligned_special_vectors,
        pep_context=ctx_prf,
        resolve_parameters=pm.resolve_parameters,
    )
    labels = [str(v) for v in best_vectors]
    print(f"V_{idx - 1}:")
    pf.pprint_labeled_matrix(best_coefficients, labels, labels)

V_1:


<IPython.core.display.Math object>

V_2:


<IPython.core.display.Math object>

V_3:


<IPython.core.display.Math object>

V_4:


<IPython.core.display.Math object>

### Consider the translated Lyapunov function $\widetilde V_k = V_k + \tau\|x_0-x_\star\|^2$.
### Then, we re-run the Steps 2-4 to verify that this is a desirable translation.

In [15]:
lyap = [V + tau_sol * (x_0 - x_star) ** 2 for V in V_raw]

for idx, V in enumerate(lyap):
    V_eval = pm.eval_scalar(V)
    rank = np.linalg.matrix_rank(V_eval.inner_prod_coords.astype(float), tol=1e-6)
    print(f"tilde V_{idx - 1}: rank={rank}")

tilde V_-1: rank=1
tilde V_0: rank=2
tilde V_1: rank=2
tilde V_2: rank=2
tilde V_3: rank=2
tilde V_4: rank=2
tilde V_5: rank=0


In [16]:
for idx in range(len(lyap)):
    aligned_special_vectors = lu.vectors_in_column_space(
        lyap[idx],
        lyap_basis_candidate,
        ctx_prf,
        resolve_parameters=pm.resolve_parameters,
        rtol=1e-5,
        atol=1e-5,
    )
    print(f"tilde V_{idx - 1}:")
    pf.pprint_labeled_vector(
        pm.eval_scalar(lyap[idx]).func_coords.astype(float),
        scalar_labels,
    )
    if not aligned_special_vectors:
        print("No candidate vector detected")
        continue
    else:
        best_vectors, best_coefficients = lu.find_basis_with_sparsest_coefficients(
            lyap[idx],
            aligned_special_vectors,
            pep_context=ctx_prf,
            resolve_parameters=pm.resolve_parameters,
        )
        labels = [str(v) for v in best_vectors]
        pf.pprint_labeled_matrix(best_coefficients, labels, labels)

tilde V_-1:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

tilde V_0:


<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

tilde V_5:


<IPython.core.display.Math object>

No candidate vector detected


### We can hypothesize the form $\widetilde V_k = a_k (f(x_k) - f(x_\star)) + b_k \|\nabla f(x_k)\|^2 + c_k \|x_{k+1} - x_\star\|^2$

---

## Step 5. Analytic proof

### We use the analytic forms 
$$
    \lambda_{i-1,i}=\frac{i}{2N+1-i}, \qquad \lambda_{\star,i}=\begin{cases}
\lambda_{0,1}, & i=0,\\
\lambda_{i,i+1}-\lambda_{i-1,i}, & 1\le i\le N-1,\\
1-\lambda_{N-1,N}, & i=N
\end{cases}
$$
$$
    s_i=\frac{1}{L}\nabla f(x_i)-\frac{1}{2N-i+1}(x_i-x_\star),
    \qquad i=0,\ldots,N.
$$
### to determine $a_k, b_k, c_k, a_{k+1}, b_{k+1}, c_{k+1}, \alpha_{k+1}$ from the equation
$$
    \widetilde{V}_k - \widetilde{V}_{k+1} = -\lambda_{k,k+1} \left( f(x_{k+1})-f(x_k) + \langle \nabla f(x_{k+1}) , x_{k} - x_{k+1} \rangle + \frac{1}{2 L} \| \nabla f(x_{k}) - \nabla f(x_{k+1}) \|^2 \right)  
    - \lambda_{\star,k+1} \left( f(x_{k+1})-f(x_{\star}) + \langle \nabla f(x_{k+1}) , x_{\star} - x_{k+1} \rangle + \frac{1}{2 L} \| \nabla f(x_{k+1}) \|^2 \right) + \alpha_{k+1} \left\|\frac{1}{L}\nabla f(x_{k+1}) - \frac{1}{2N-k} (x_{k+1} - x_\star)\right\|^2 
$$

In [17]:
def lambda_adj_formula(i, N):
    if 0 <= i <= N - 1:
        return sp.S(i + 1) / sp.S(2 * N - i)
    return sp.S(0)


def lambda_star_formula(i, N):
    if i == 0:
        return lambda_adj_formula(0, N)
    if 1 <= i <= N - 1:
        return lambda_adj_formula(i, N) - lambda_adj_formula(i - 1, N)
    if i == N:
        return sp.S(1) - lambda_adj_formula(N - 1, N)
    return sp.S(0)


def lambda_formula(tag_i, tag_j, N):
    i = tag_to_index(tag_i, N)
    j = tag_to_index(tag_j, N)
    if tag_i == "x_star" and 0 <= j <= N:
        return lambda_star_formula(j, N)
    if 0 <= i <= N - 1 and j == i + 1:
        return lambda_adj_formula(i, N)
    return sp.S(0)


def exact_simplify(expr):
    return sp.factor(sp.together(sp.simplify(sp.nsimplify(expr))))


def exact_matrix(array):
    return sp.Matrix(array).applyfunc(exact_simplify)


def solve_symbolic_identity(diff, ctx, resolve_parameters, unknowns):
    pm_check = pf.ExpressionManager(ctx, resolve_parameters=resolve_parameters)
    matrix_part = exact_matrix(pm_check.eval_scalar(diff).inner_prod_coords)
    func_part = exact_matrix(pm_check.eval_scalar(diff).func_coords)
    equations = [entry for entry in list(matrix_part) + list(func_part) if entry != 0]
    solution_set = sp.linsolve(equations, unknowns)
    solution_tuple = next(iter(solution_set))
    solution = {
        unknown: exact_simplify(expression)
        for unknown, expression in zip(unknowns, solution_tuple)
    }
    return matrix_part, func_part, solution_set, solution


def residual_after_substitution(matrix_part, func_part, solution):
    matrix_residual = exact_matrix(matrix_part.subs(solution))
    func_residual = exact_matrix(func_part.subs(solution))
    return matrix_residual, func_residual


def display_solution(title, solution):
    print(title)
    display(
        Math(
            r"\begin{aligned}"
            + r"\\".join(
                sp.latex(sp.Eq(unknown, exact_simplify(expression)))
                for unknown, expression in solution.items()
            )
            + r"\end{aligned}"
        )
    )


def display_verification(matrix_residual, func_residual):
    print("quadratic residual:")
    display(matrix_residual)
    print("function-value residual:")
    display(func_residual)
    print(
        "identity verified?",
        matrix_residual == sp.zeros(*matrix_residual.shape)
        and func_residual == sp.zeros(*func_residual.shape),
    )

### Case 1: $k=-1$

### Here we have $\widetilde V_{-1}=\frac{L}{4N+2}\|x_0-x_\star\|^2$ and solve $\widetilde V_{-1}-\widetilde V_0 = -\lambda_{\star,0} I_{\star,0}+\alpha_0\|s_0\|^2$ for $a_0,b_0,c_0,\alpha_0$.

In [18]:
ctx_gd_first = pf.PEPContext("gd_first_endpoint_solve").set_as_current()
N_param = pf.Parameter("N")
f.set_stationary_point("x_star")
x_star_first = ctx_gd_first["x_star"]
x0_first = pf.Vector(is_basis=True, tags=["x_0"])
x1_first = x0_first - 1 / L * f.grad(x0_first)
x1_first.add_tag("x_1")

a_0 = pf.Parameter("a_0")
b_0 = pf.Parameter("b_0")
c_0 = pf.Parameter("c_0")
alpha_0 = pf.Parameter("alpha_0")

V_minus_1 = L / (4 * N_param + 2) * (x0_first - x_star_first) ** 2
V_0_ansatz = (
    a_0 * (f(x0_first) - f(x_star_first))
    + b_0 * f.grad(x0_first) ** 2
    + c_0 * (x1_first - x_star_first) ** 2
)
lambda_star_0 = sp.S(1) / (2 * N_param)
s_0 = 1 / L * f.grad(x0_first) - 1 / (2 * N_param + 1) * (x0_first - x_star_first)

first_identity_residual = (
    V_minus_1
    - V_0_ansatz
    + lambda_star_0 * f.interp_ineq("x_star", "x_0")
    - alpha_0 * s_0**2
)

N_sp, L_sp = sp.symbols("N L", positive=True)
a_0_sp, b_0_sp, c_0_sp = sp.symbols("a_0 b_0 c_0")
alpha_0_sp = sp.Symbol(r"\alpha_0")
first_unknowns = (a_0_sp, b_0_sp, c_0_sp, alpha_0_sp)
first_matrix, first_func, first_solution_set, first_solution = solve_symbolic_identity(
    first_identity_residual,
    ctx_gd_first,
    {
        "N": N_sp,
        "L": L_sp,
        "a_0": a_0_sp,
        "b_0": b_0_sp,
        "c_0": c_0_sp,
        "alpha_0": alpha_0_sp,
    },
    first_unknowns,
)

display_solution("Solution for the first transition:", first_solution)
alpha_0_formula = exact_simplify(
    L_sp / sp.S(2) * (2 * N_sp + 1) / (2 * N_sp) * (sp.S(1) / (2 * N_sp) + sp.S(0))
)
print(
    "alpha_0 matches closed-form alpha formula?",
    exact_simplify(first_solution[alpha_0_sp] - alpha_0_formula) == 0,
)
first_matrix_residual, first_func_residual = residual_after_substitution(
    first_matrix, first_func, first_solution
)
display_verification(first_matrix_residual, first_func_residual)

Solution for the first transition:


<IPython.core.display.Math object>

alpha_0 matches closed-form alpha formula? True
quadratic residual:


Matrix([
[0, 0, 0],
[0, 0, 0],
[0, 0, 0]])

function-value residual:


Matrix([
[0],
[0]])

identity verified? True


### Case 2: $0\le k\le N-2$

### We solve $\widetilde V_k-\widetilde V_{k+1} = -\lambda_{k,k+1} I_{k,k+1} - \lambda_{\star,k+1} I_{\star,k+1} + \alpha_{k+1}\|s_{k+1}\|^2$. The linear system yields a rank-1 solution family, and consistency of the formula for $b_k, b_{k+1}$ uniquely determines the closed form for all coefficients.

In [19]:
ctx_gd_interior = pf.PEPContext("gd_interior_solve").set_as_current()
N_param = pf.Parameter("N")
k_param = pf.Parameter("k")
f.set_stationary_point("x_star")
x_star_int = ctx_gd_interior["x_star"]
x_k = pf.Vector(is_basis=True, tags=["x_k"])
x_k1 = x_k - 1 / L * f.grad(x_k)
x_k1.add_tag("x_{k+1}")
x_k2 = x_k1 - 1 / L * f.grad(x_k1)
x_k2.add_tag("x_{k+2}")

a_k = pf.Parameter("a_k")
b_k = pf.Parameter("b_k")
c_k = pf.Parameter("c_k")
a_k1 = pf.Parameter("a_{k+1}")
b_k1 = pf.Parameter("b_{k+1}")
c_k1 = pf.Parameter("c_{k+1}")
alpha_k1 = pf.Parameter("alpha_{k+1}")

V_k_ansatz = (
    a_k * (f(x_k) - f(x_star_int))
    + b_k * f.grad(x_k) ** 2
    + c_k * (x_k1 - x_star_int) ** 2
)
V_k1_ansatz = (
    a_k1 * (f(x_k1) - f(x_star_int))
    + b_k1 * f.grad(x_k1) ** 2
    + c_k1 * (x_k2 - x_star_int) ** 2
)

lambda_k_k1 = (k_param + 1) / (2 * N_param - k_param)
lambda_star_k1 = (k_param + 2) / (2 * N_param - k_param - 1) - lambda_k_k1
s_k1 = 1 / L * f.grad(x_k1) - 1 / (2 * N_param - k_param) * (x_k1 - x_star_int)

interior_identity_residual = (
    V_k_ansatz
    - V_k1_ansatz
    + lambda_k_k1 * f.interp_ineq("x_k", "x_{k+1}")
    + lambda_star_k1 * f.interp_ineq("x_star", "x_{k+1}")
    - alpha_k1 * s_k1**2
)

N_sp, k_sp, L_sp = sp.symbols("N k L", positive=True)
(
    a_k_sp,
    b_k_sp,
    c_k_sp,
    a_k1_sp,
    b_k1_sp,
    c_k1_sp,
) = sp.symbols("a_k b_k c_k a_{k+1} b_{k+1} c_{k+1}")
alpha_k1_sp = sp.Symbol(r"\alpha_{k+1}")
interior_unknowns = (
    a_k_sp,
    b_k_sp,
    c_k_sp,
    a_k1_sp,
    b_k1_sp,
    c_k1_sp,
    alpha_k1_sp,
)
interior_matrix, interior_func, interior_solution_set, interior_param_solution = (
    solve_symbolic_identity(
        interior_identity_residual,
        ctx_gd_interior,
        {
            "N": N_sp,
            "k": k_sp,
            "L": L_sp,
            "a_k": a_k_sp,
            "b_k": b_k_sp,
            "c_k": c_k_sp,
            "a_{k+1}": a_k1_sp,
            "b_{k+1}": b_k1_sp,
            "c_{k+1}": c_k1_sp,
            "alpha_{k+1}": alpha_k1_sp,
        },
        interior_unknowns,
    )
)

display_solution(
    "Parametric solution for the interior transition:", interior_param_solution
)

b_k1_shifted = -(k_sp + 2) / (2 * L_sp * (2 * N_sp - k_sp - 1))
alpha_k1_closed = sp.solve(
    sp.Eq(interior_param_solution[b_k1_sp], b_k1_shifted), alpha_k1_sp
)[0]
alpha_k1_closed = exact_simplify(alpha_k1_closed)

interior_solution = {
    unknown: exact_simplify(expression.subs(alpha_k1_sp, alpha_k1_closed))
    for unknown, expression in interior_param_solution.items()
}

display_solution("Closed form selected by consistency of b_{k+1}:", interior_solution)
alpha_k1_formula = exact_simplify(
    L_sp
    / sp.S(2)
    * (2 * N_sp - k_sp)
    / (2 * N_sp - k_sp - 1)
    * ((k_sp + 2) / (2 * N_sp - k_sp - 1) + (k_sp + 1) / (2 * N_sp - k_sp))
)
print(
    "alpha_{k+1} matches closed-form alpha formula?",
    exact_simplify(interior_solution[alpha_k1_sp] - alpha_k1_formula) == 0,
)
interior_matrix_residual, interior_func_residual = residual_after_substitution(
    interior_matrix, interior_func, interior_solution
)
display_verification(interior_matrix_residual, interior_func_residual)

Parametric solution for the interior transition:


<IPython.core.display.Math object>

Closed form selected by consistency of b_{k+1}:


<IPython.core.display.Math object>

alpha_{k+1} matches closed-form alpha formula? True
quadratic residual:


Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0]])

identity verified? True


### Case 3: $k=N-1$

### We check this case separately because the formula for $\lambda_{\star,N}$ is different from Case 2.

In [20]:
ctx_gd_last = pf.PEPContext("gd_last_endpoint_solve").set_as_current()
N_param = pf.Parameter("N")
f.set_stationary_point("x_star")
x_star_last = ctx_gd_last["x_star"]
x_Nm1 = pf.Vector(is_basis=True, tags=["x_{N-1}"])
x_N = x_Nm1 - 1 / L * f.grad(x_Nm1)
x_N.add_tag("x_N")

a_Nm1 = pf.Parameter("a_{N-1}")
b_Nm1 = pf.Parameter("b_{N-1}")
c_Nm1 = pf.Parameter("c_{N-1}")
alpha_N = pf.Parameter("alpha_N")

V_Nm1_ansatz = (
    a_Nm1 * (f(x_Nm1) - f(x_star_last))
    + b_Nm1 * f.grad(x_Nm1) ** 2
    + c_Nm1 * (x_N - x_star_last) ** 2
)
V_N_terminal = f(x_N) - f(x_star_last)

lambda_Nm1_N = N_param / (N_param + 1)
lambda_star_N = sp.S(1) / (N_param + 1)
s_N = 1 / L * f.grad(x_N) - 1 / (N_param + 1) * (x_N - x_star_last)

last_identity_residual = (
    V_Nm1_ansatz
    - V_N_terminal
    + lambda_Nm1_N * f.interp_ineq("x_{N-1}", "x_N")
    + lambda_star_N * f.interp_ineq("x_star", "x_N")
    - alpha_N * s_N**2
)

N_sp, L_sp = sp.symbols("N L", positive=True)
a_Nm1_sp, b_Nm1_sp, c_Nm1_sp = sp.symbols("a_{N-1} b_{N-1} c_{N-1}")
alpha_N_sp = sp.Symbol(r"\alpha_N")
last_unknowns = (a_Nm1_sp, b_Nm1_sp, c_Nm1_sp, alpha_N_sp)
last_matrix, last_func, last_solution_set, last_solution = solve_symbolic_identity(
    last_identity_residual,
    ctx_gd_last,
    {
        "N": N_sp,
        "L": L_sp,
        "a_{N-1}": a_Nm1_sp,
        "b_{N-1}": b_Nm1_sp,
        "c_{N-1}": c_Nm1_sp,
        "alpha_N": alpha_N_sp,
    },
    last_unknowns,
)

display_solution("Solution for the terminal transition:", last_solution)
alpha_N_formula = exact_simplify(
    L_sp / sp.S(2) * (N_sp + 1) / N_sp * (sp.S(0) + N_sp / (N_sp + 1))
)
print(
    "alpha_N matches closed-form alpha formula?",
    exact_simplify(last_solution[alpha_N_sp] - alpha_N_formula) == 0,
)
last_matrix_residual, last_func_residual = residual_after_substitution(
    last_matrix, last_func, last_solution
)
display_verification(last_matrix_residual, last_func_residual)

Solution for the terminal transition:


<IPython.core.display.Math object>

alpha_N matches closed-form alpha formula? True
quadratic residual:


Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

function-value residual:


Matrix([
[0],
[0],
[0]])

identity verified? True


## Conclusion

### Summarizing the above, we obtain 
$$
\widetilde V_k=
\frac{k+1}{2N-k}\left(f(x_k)-f(x_\star)-\frac{1}{2L}\|\nabla f(x_k)\|^2\right)
+\frac{L}{2}\frac{2N-2k-1}{(2N-k)^2}\|x_{k+1}-x_\star\|^2
$$
### for $0\le k\le N-1$, together with $\widetilde V_{-1}=\frac{L}{4N+2}\|x_0-x_\star\|^2$ and $\widetilde V_N=f(x_N)-f(x_\star)$, is a Lyapunov function.